# Suite TSPRT — Printing TypeScript

The standards print TypeScript trees as source text in one fixed layout (`TypeScript/_Printer`). Trees built by hand
carry no parentheses, so the printer adds those the grammar needs. Each case prints hand-built trees and checks the
text, which parses back to the same tree but for the parentheses added.

In [ ]:
import { Errors, Syntax as F } from "@mbse/programs/Framework";
import { ECMAScript2025, Syntax as S, TypeScript59 } from "@mbse/programs/TypeScript";
import { Printer } from "@mbse/programs/TypeScript/_Printer";

import { assert, equal, raises } from "./support.js";

const { PrintError } = Errors;
const TS = TypeScript59.STANDARD;
const TSX = TypeScript59.JSX;
const n = (name: string) => new S.Identifier({ name });
const lit = (raw: string) => new S.Literal({ raw });
const ref = (name: string) => new S.TSTypeReference({ typeName: n(name) });
const binary = (left: F.SyntaxNode, operator: string, right: F.SyntaxNode): F.SyntaxNode => ["&&", "||", "??"].includes(operator)
  ? new S.LogicalExpression({ left, operator: operator as S.LogicalOperator, right })
  : new S.BinaryExpression({ left, operator: operator as S.BinaryOperator, right });

/** `node` without the parentheses the source wrote. */
function strip(node: any): any {
  if (node instanceof S.ParenthesizedExpression) return strip(node.expression);
  if (node instanceof S.TSParenthesizedType) return strip(node.typeAnnotation);
  if (Array.isArray(node)) return node.map(strip);
  if (node instanceof F.SyntaxNode) {
    for (const prop of node.kind().PROPERTIES) (node as any)[prop.name] = strip(node.get(prop.name));
  }
  return node;
}

/** `node` with each place holding a node of its own: hand-built trees here reuse a few names. */
function unshare(node: any): any {
  if (Array.isArray(node)) return node.map(unshare);
  if (node instanceof F.SyntaxNode) {
    const kind = node.kind() as unknown as new (values: object) => F.SyntaxNode;
    return new kind(Object.fromEntries(node.kind().PROPERTIES.map((f) => [f.name, unshare(node.get(f.name))])));
  }
  return node;
}

/** `node` prints as `expected`, which parses back to `node` but for parentheses. */
function prints(original: F.SyntaxNode, expected: string, standard = TS): void {
  const node = unshare(original);
  const text = standard.print(node);
  assert(text === expected, text);
  if (node instanceof S.Expression && !(node instanceof S.Identifier)) {
    const back = strip((standard.parse(`(${expected});`).body[0] as S.ExpressionStatement).expression);
    assert(F.same(back, node), `${back} != ${node}`);
  }
}

## TSPRT-01 · Precedence: operands are parenthesized where they bind less than their place needs

In [ ]:
const [a, b, c] = [n("a"), n("b"), n("c")];
prints(binary(binary(a, "+", b), "*", c), "(a + b) * c");
prints(binary(a, "-", binary(b, "-", c)), "a - (b - c)");
prints(binary(binary(a, "-", b), "-", c), "a - b - c");
prints(binary(a, "**", binary(b, "**", c)), "a ** b ** c");
prints(binary(binary(a, "**", b), "**", c), "(a ** b) ** c");
prints(binary(new S.UnaryExpression({ operator: "-", argument: a }), "**", b), "(-a) ** b");
prints(binary(new S.AwaitExpression({ argument: a }), "**", b), "(await a) ** b");
prints(binary(binary(a, "??", b), "||", c), "(a ?? b) || c");
prints(binary(a, "??", binary(b, "&&", c)), "a ?? (b && c)");
prints(binary(binary(a, "||", b), "??", c), "(a || b) ?? c");
prints(binary(binary(a, "&&", b), "||", c), "a && b || c");
prints(binary(new S.PrivateIdentifier({ name: "x" }), "in", a), "#x in a");
prints(new S.ConditionalExpression({ test: new S.ConditionalExpression({ test: a, consequent: b, alternate: c }),
  consequent: b, alternate: new S.SequenceExpression({ expressions: [a, b] }) }), "(a ? b : c) ? b : (a, b)");
prints(new S.AssignmentExpression({ left: a, operator: "=", right: new S.AssignmentExpression({ left: b, operator: "+=",
  right: c }) }), "a = b += c");
prints(new S.AssignmentExpression({ left: new S.ObjectPattern({ properties: [new S.Property({ propertyKind: "init", key: a,
  value: n("a"), shorthand: true })] }), operator: "=", right: b }), "{ a } = b");
prints(new S.UnaryExpression({ operator: "-", argument: new S.UnaryExpression({ operator: "-", argument: a }) }), "- -a");
prints(new S.UnaryExpression({ operator: "+", argument: new S.UpdateExpression({ operator: "++", prefix: true,
  argument: a }) }), "+ ++a");
prints(new S.UnaryExpression({ operator: "-", argument: new S.UnaryExpression({ operator: "+", argument: a }) }), "-+a");
prints(new S.UnaryExpression({ operator: "typeof", argument: binary(a, "+", b) }), "typeof (a + b)");
prints(new S.UnaryExpression({ operator: "!", argument: new S.UnaryExpression({ operator: "!", argument: a }) }), "!!a");
prints(new S.UpdateExpression({ operator: "++", prefix: false, argument: new S.MemberExpression({ object: a,
  property: b }) }), "a.b++");
prints(new S.UpdateExpression({ operator: "--", prefix: true, argument: a }), "--a");
prints(new S.MemberExpression({ object: binary(a, "+", b), property: c }), "(a + b).c");
prints(new S.MemberExpression({ object: lit("1"), property: n("toString") }), "(1).toString");
prints(new S.MemberExpression({ object: lit("1.5"), property: n("toFixed") }), "1.5.toFixed");
prints(new S.MemberExpression({ object: new S.NewExpression({ callee: a }), property: b }), "new a().b");
prints(new S.ChainExpression({ expression: new S.MemberExpression({ object: a, property: binary(b, "+", c),
  computed: true, optional: true }) }), "a?.[b + c]");
prints(new S.CallExpression({ callee: new S.ArrowFunctionExpression({ body: a }), arguments: [
  new S.SequenceExpression({ expressions: [a, b] })] }), "(() => a)((a, b))");
prints(new S.NewExpression({ callee: new S.CallExpression({ callee: a }) }), "new (a())()");
prints(new S.NewExpression({ callee: new S.MemberExpression({ object: new S.CallExpression({ callee: a }),
  property: b }) }), "new (a().b)()");
prints(new S.NewExpression({ callee: new S.MemberExpression({ object: new S.TSNonNullExpression({ expression: a }),
  property: b }) }), "new (a!.b)()");
prints(new S.NewExpression({ callee: new S.TaggedTemplateExpression({ tag: a, quasi: new S.TemplateLiteral({
  quasis: [new S.TemplateElement({ raw: "x", tail: true })] }) }) }), "new a`x`()");
prints(new S.NewExpression({ callee: new S.NewExpression({ callee: a }) }), "new (new a())()");
prints(new S.TSAsExpression({ expression: binary(a, "??", b), typeAnnotation: ref("T") }), "(a ?? b) as T");
prints(binary(new S.TSAsExpression({ expression: a, typeAnnotation: ref("T") }), "+", b), "(a as T) + b");
prints(new S.TSNonNullExpression({ expression: new S.AwaitExpression({ argument: a }) }), "(await a)!");
prints(new S.TSTypeAssertion({ typeAnnotation: ref("T"), expression: binary(a, "+", b) }), "<T>(a + b)");
prints(new S.AwaitExpression({ argument: new S.TSTypeAssertion({ typeAnnotation: ref("T"), expression: a }) }),
  "await <T>a");
prints(new S.YieldExpression({ argument: new S.SequenceExpression({ expressions: [a, b] }) }), "yield (a, b)");
prints(new S.SpreadElement({ argument: new S.SequenceExpression({ expressions: [a, b] }) }), "...(a, b)");
prints(new S.ArrayExpression({ elements: [new S.Elision(), a, new S.Elision(), new S.Elision()] }), "[, a,,,]");
prints(new S.ArrayExpression({ elements: [a, new S.Elision(), lit("', ,'")] }), "[a,, ', ,']");
console.log("ok");

## TSPRT-02 · Where an expression starts a statement, an arrow's body, a `for` or a default export

In [ ]:
const statement = (expression: F.SyntaxNode) => new S.ExpressionStatement({ expression });
const empty = () => new S.EmptyStatement();

prints(statement(new S.ObjectExpression()), "({});");
prints(statement(new S.CallExpression({ callee: new S.FunctionExpression({ body: new S.BlockStatement() }) })),
  "(function() {}());");
prints(statement(new S.MemberExpression({ object: new S.ClassExpression({ body: new S.ClassBody() }), property: a })),
  "(class {}.a);");
prints(statement(new S.AssignmentExpression({ left: new S.ObjectPattern(), operator: "=", right: a })), "({} = a);");
prints(statement(new S.AssignmentExpression({ left: new S.MemberExpression({ object: n("let"), property: a,
  computed: true }), operator: "=", right: b })), "(let[a] = b);");
prints(statement(new S.UpdateExpression({ operator: "++", argument: new S.MemberExpression({
  object: new S.ObjectExpression(), property: a }) })), "({}.a++);");
prints(statement(new S.TSAsExpression({ expression: new S.ObjectExpression(), typeAnnotation: ref("T") })),
  "({} as T);");
prints(statement(new S.ConditionalExpression({ test: new S.TaggedTemplateExpression({
  tag: new S.FunctionExpression({ body: new S.BlockStatement() }), quasi: new S.TemplateLiteral({
    quasis: [new S.TemplateElement({ raw: "", tail: true })] }) }), consequent: a, alternate: b })),
"(function() {}`` ? a : b);");
prints(new S.ArrowFunctionExpression({ body: new S.ObjectExpression() }), "() => ({})");
prints(new S.ArrowFunctionExpression({ body: new S.MemberExpression({ object: new S.ObjectExpression(), property: a }) }),
  "() => ({}.a)");
prints(new S.ArrowFunctionExpression({ body: new S.SequenceExpression({ expressions: [a, b] }) }), "() => (a, b)");
prints(new S.ArrowFunctionExpression({ params: [a], body: new S.ArrowFunctionExpression({ body: a }) }),
  "(a) => () => a");
prints(new S.ForStatement({ init: new S.VariableDeclaration({ declarationKind: "let", declarations: [
  new S.VariableDeclarator({ id: a, init: binary(lit("'x'"), "in", b) })] }), body: empty() }),
"for (let a = ('x' in b);;)\n    ;");
for (const init of [new S.ConditionalExpression({ test: binary(a, "in", b), consequent: a, alternate: b }),
  new S.SequenceExpression({ expressions: [a, binary(a, "in", b)] }),
  new S.TSAsExpression({ expression: binary(a, "in", b), typeAnnotation: ref("T") }),
  new S.ArrowFunctionExpression({ body: binary(a, "in", b) }), binary(a, "&&", binary(a, "in", b)),
  new S.AssignmentExpression({ left: a, operator: "=", right: binary(a, "in", b) }),
  new S.ConditionalExpression({ test: a, consequent: b, alternate: binary(a, "in", b) })]) {
  const text = TS.print(unshare(new S.ForStatement({ init, body: empty() })));
  assert(text.startsWith("for ((") && text.includes(";;)"), text);
}
assert(TS.print(unshare(new S.ForStatement({ init: binary(a, "+", b), body: empty() }))) === "for (a + b;;)\n    ;");
prints(new S.ForOfStatement({ left: n("async"), right: a, body: empty() }), "for ((async) of a)\n    ;");
prints(new S.ForOfStatement({ isAwait: true, left: n("async"), right: a, body: empty() }), "for await (async of a)\n    ;");
prints(new S.ForOfStatement({ left: a, right: new S.SequenceExpression({ expressions: [a, b] }), body: empty() }),
  "for (a of (a, b))\n    ;");
prints(new S.ForInStatement({ left: new S.MemberExpression({ object: a, property: b }), right: c,
  body: new S.BlockStatement() }), "for (a.b in c) {}");
prints(new S.ExportDefaultDeclaration({ declaration: new S.FunctionExpression({ body: new S.BlockStatement() }) }),
  "export default (function() {});");
prints(new S.ExportDefaultDeclaration({ declaration: new S.CallExpression({ callee: new S.ClassExpression({
  body: new S.ClassBody() }) }) }), "export default (class {}());");
prints(new S.ExportDefaultDeclaration({ declaration: new S.SequenceExpression({ expressions: [a, b] }) }),
  "export default (a, b);");
prints(new S.ExportDefaultDeclaration({ declaration: new S.ArrowFunctionExpression({ body: a }) }),
  "export default () => a;");
console.log("ok");

## TSPRT-03 · Types: parentheses where a function, conditional, union or operator type is an operand

In [ ]:
const [T, U] = [ref("T"), ref("U")];
const annotation = (typeAnnotation: F.SyntaxNode) => new S.TSTypeAnnotation({ typeAnnotation });
const fn = new S.TSFunctionType({ returnType: annotation(T) });
const union = new S.TSUnionType({ types: [T, U] });
const conditional = new S.TSConditionalType({ checkType: T, extendsType: U, trueType: T, falseType: U });
prints(new S.TSArrayType({ elementType: union }), "(T | U)[]");
prints(new S.TSUnionType({ types: [fn, T] }), "(() => T) | T");
prints(new S.TSIntersectionType({ types: [union, T] }), "(T | U) & T");
prints(new S.TSUnionType({ types: [new S.TSIntersectionType({ types: [T, U] }), T] }), "T & U | T");
prints(new S.TSTypeOperator({ operator: "keyof", typeAnnotation: union }), "keyof (T | U)");
prints(new S.TSArrayType({ elementType: new S.TSTypeOperator({ operator: "keyof", typeAnnotation: T }) }), "(keyof T)[]");
prints(new S.TSIndexedAccessType({ objectType: union, indexType: union }), "(T | U)[T | U]");
prints(new S.TSOptionalType({ typeAnnotation: union }), "(T | U)?");
prints(new S.TSConditionalType({ checkType: conditional, extendsType: conditional, trueType: conditional,
  falseType: conditional }),
"(T extends U ? T : U) extends (T extends U ? T : U) ? T extends U ? T : U : T extends U ? T : U");
prints(new S.TSConditionalType({ checkType: fn, extendsType: fn, trueType: T, falseType: U }),
  "(() => T) extends () => T ? T : U");
const returnsConditional = new S.TSFunctionType({ returnType: annotation(conditional) });
prints(new S.TSConditionalType({ checkType: T, extendsType: returnsConditional, trueType: T, falseType: U }),
  "T extends (() => T extends U ? T : U) ? T : U");
prints(new S.TSConditionalType({ checkType: T, extendsType: new S.TSConstructorType(), trueType: T, falseType: U }),
  "T extends new () ? T : U");
prints(new S.TSTypeOperator({ operator: "readonly", typeAnnotation: new S.TSArrayType({ elementType: T }) }),
  "readonly T[]");
prints(new S.TSTypeOperator({ operator: "unique", typeAnnotation: new S.TSSymbolKeyword() }), "unique symbol");
prints(new S.TSInferType({ typeParameter: new S.TSTypeParameter({ name: n("X"), constraint: union }) }),
  "infer X extends T | U");
prints(new S.TSArrayType({ elementType: new S.TSInferType({ typeParameter: new S.TSTypeParameter({ name: n("X") }) }) }),
  "(infer X)[]");
console.log("ok");

## TSPRT-04 · Every kind prints on its own, as the text it stands for

In [ ]:
const parameters = (...params: F.SyntaxNode[]) => new S.TSTypeParameterInstantiation({ params });
const texts: [F.SyntaxNode, string][] = [
  [new S.Comment({ text: " a" }), "// a"], [new S.Comment({ block: true, text: " a " }), "/* a */"],
  [new S.ThisExpression(), "this"], [new S.Super(), "super"], [new S.Elision(), ""],
  [new S.TemplateElement({ raw: "a" }), "a"], [new S.ParenthesizedExpression({ expression: a }), "(a)"],
  [new S.MetaProperty({ meta: n("new"), property: n("target") }), "new.target"],
  [new S.ChainExpression({ expression: new S.CallExpression({ callee: a, optional: true }) }), "a?.()"],
  [new S.ImportExpression({ source: lit("'m'"), options: a }), "import('m', a)"],
  [new S.ImportExpression({ source: lit("'m'"), phase: "defer" }), "import.defer('m')"],
  [new S.TSInstantiationExpression({ expression: a, typeArguments: parameters(T) }), "a<T>"],
  [new S.TSSatisfiesExpression({ expression: a, typeAnnotation: T }), "a satisfies T"],
  [new S.Decorator({ expression: new S.CallExpression({ callee: a }) }), "@a()"],
  [new S.ImportAttribute({ key: n("type"), value: lit("'json'") }), "type: 'json'"],
  [new S.ImportSpecifier({ importKind: "value", imported: a, local: n("a") }), "a"],
  [new S.ImportSpecifier({ importKind: "type", imported: a, local: b }), "type a as b"],
  [new S.ImportDefaultSpecifier({ local: a }), "a"], [new S.ImportNamespaceSpecifier({ local: a }), "* as a"],
  [new S.ExportSpecifier({ exportKind: "value", local: a, exported: n("a") }), "a"],
  [new S.ExportSpecifier({ exportKind: "type", local: a, exported: b }), "type a as b"],
  [new S.TSExternalModuleReference({ expression: lit("'m'") }), "require('m')"],
  [annotation(T), "T"],
  [new S.TSTypeParameter({ const: true, isIn: true, isOut: true, name: n("X"), constraint: T, default: U }),
    "const in out X extends T = U"],
  [new S.TSTypeParameterDeclaration({ params: [new S.TSTypeParameter({ name: n("X") })] }), "<X>"],
  [new S.TSInterfaceHeritage({ expression: new S.MemberExpression({ object: a, property: b }) }), "a.b"],
  [new S.TSClassImplements({ expression: a, typeArguments: parameters(T) }), "a<T>"],
  [new S.TSEnumMember({ id: lit("'a'"), initializer: lit("1") }), "'a' = 1"],
  [new S.TSEnumMember({ id: a, computed: true }), "[a]"],
  [new S.TSPropertySignature({ readonly: true, key: a, optional: true, typeAnnotation: annotation(T) }), "readonly a?: T;"],
  [new S.TSPropertySignature({ key: a, computed: true }), "[a];"],
  [new S.TSMethodSignature({ methodKind: "get", key: a }), "get a();"],
  [new S.TSCallSignatureDeclaration({ params: [a] }), "(a);"],
  [new S.TSConstructSignatureDeclaration({ typeParameters: new S.TSTypeParameterDeclaration({
    params: [new S.TSTypeParameter({ name: n("X") })] }) }), "new <X>();"],
  [new S.TSIndexSignature({ accessibility: "private", static: true, readonly: true, parameters: [
    new S.Identifier({ name: "k", typeAnnotation: annotation(new S.TSStringKeyword()) })] }),
  "private static readonly [k: string];"],
  [new S.TSIntrinsicKeyword(), "intrinsic"], [new S.TSThisType(), "this"],
  [new S.TSQualifiedName({ left: a, right: b }), "a.b"], [new S.TSParenthesizedType({ typeAnnotation: union }), "(T | U)"],
  [new S.TSLiteralType({ literal: lit("'a'") }), "'a'"],
  [new S.TSTemplateLiteralType({ quasis: [new S.TemplateElement({ raw: "a" }), new S.TemplateElement({ raw: "b",
    tail: true })], types: [T] }), "`a${T}b`"],
  [new S.TSTupleType({ elementTypes: [new S.TSNamedTupleMember({ label: a, optional: true, elementType: T }),
    new S.TSRestType({ typeAnnotation: new S.TSArrayType({ elementType: U }) })] }), "[a?: T, ...U[]]"],
  [new S.TSTypeLiteral(), "{}"], [new S.TSMappedType({ key: n("K"), constraint: T }), "{ [K in T] }"],
  [new S.TSTypeQuery({ exprName: new S.TSImportType({ source: lit("'m'") }), typeArguments: parameters(T) }),
    "typeof import('m')<T>"],
  [new S.TSImportType({ source: lit("'m'"), options: new S.ObjectExpression(), qualifier: a }), "import('m', {}).a"],
  [new S.TSTypePredicate({ asserts: true, parameterName: new S.TSThisType() }), "asserts this"],
  [new S.TSTypePredicate({ parameterName: a, typeAnnotation: annotation(T) }), "a is T"],
  [new S.TSConstructorType({ abstract: true, params: [a], returnType: annotation(T) }), "abstract new (a) => T"],
  [new S.TSTypeOperator({ operator: "keyof" }), "keyof"],
  [new S.ObjectPattern({ optional: true, typeAnnotation: annotation(T) }), "{}?: T"],
  [new S.ArrayPattern({ elements: [a, new S.Elision()], decorators: [new S.Decorator({ expression: a })] }), "@a [a,,]"],
  [new S.RestElement({ argument: a, typeAnnotation: annotation(T) }), "...a: T"],
  [new S.AssignmentPattern({ left: a, right: new S.SequenceExpression({ expressions: [a, b] }) }), "a = (a, b)"],
  [new S.TSParameterProperty({ decorators: [new S.Decorator({ expression: a })], accessibility: "private",
    readonly: true, override: true, parameter: b }), "@a private override readonly b"],
  [new S.Property({ propertyKind: "init", key: a, value: new S.AssignmentPattern({ left: n("a"), right: lit("1") }),
    shorthand: true }), "a = 1"],
  [new S.Property({ propertyKind: "init", key: a, value: new S.TSEmptyBodyFunctionExpression(), method: true }), "a()"],
  [new S.Property({ propertyKind: "init", key: a, computed: true, optional: true, value: new S.FunctionExpression({
    isAsync: true, generator: true, body: new S.BlockStatement() }), method: true }), "async *[a]?() {}"],
  [new S.Property({ propertyKind: "get", key: a, value: new S.FunctionExpression({ body: new S.BlockStatement() }) }),
    "get a() {}"],
  [new S.YieldExpression({ delegate: true, argument: a }), "yield* a"], [new S.YieldExpression(), "yield"],
  [new S.ClassExpression({ decorators: [new S.Decorator({ expression: a })], id: n("C"), body: new S.ClassBody() }),
    "@a class C {}"],
  [new S.JSXIdentifier({ name: "a-b" }), "a-b"], [new S.JSXEmptyExpression(), ""],
  [new S.JSXSpreadAttribute({ argument: new S.SequenceExpression({ expressions: [a, b] }) }), "{...(a, b)}"],
  [new S.JSXSpreadChild({ expression: a }), "{...a}"], [new S.JSXText({ raw: " a " }), " a "],
  [new S.JSXAttribute({ name: new S.JSXNamespacedName({ namespace: new S.JSXIdentifier({ name: "a" }),
    name: new S.JSXIdentifier({ name: "b" }) }) }), "a:b"],
  [new S.JSXFragment({ openingFragment: new S.JSXOpeningFragment(), closingFragment: new S.JSXClosingFragment() }),
    "<></>"],
  [new S.SwitchCase({ consequent: [new S.Comment({ text: " c", trailing: true }), new S.BreakStatement()] }),
    "default: // c\n    break;"],
  [new S.MethodDefinition({ methodKind: "method", key: a, value: new S.TSEmptyBodyFunctionExpression() }), "a();"],
  [new S.TSAbstractMethodDefinition({ accessibility: "protected", methodKind: "get", key: a,
    value: new S.TSEmptyBodyFunctionExpression() }), "protected abstract get a();"],
  [new S.StaticBlock(), "static {}"], [new S.TSDeclareFunction({ id: a }), "function a();"],
  [new S.ClassDeclaration({ declare: true, abstract: true, id: n("C"), body: new S.ClassBody() }),
    "declare abstract class C {}"],
  [new S.TSInterfaceDeclaration({ declare: true, id: n("I"), body: new S.TSInterfaceBody() }), "declare interface I {}"],
  [new S.TSEnumDeclaration({ id: n("E"), body: new S.TSEnumBody() }), "enum E {}"],
  [new S.TSModuleDeclaration({ moduleKind: "global", id: n("global"), body: new S.TSModuleBlock() }), "global {}"],
  [new S.TSTypeAliasDeclaration({ declare: true, id: n("X"), typeAnnotation: T }), "declare type X = T;"],
  [new S.VariableDeclaration({ declare: true, declarationKind: "var", declarations: [new S.VariableDeclarator({ id: a })] }),
    "declare var a;"],
  [new S.ImportDeclaration({ importKind: "type", source: lit("'m'"), specifiers: [
    new S.ImportDefaultSpecifier({ local: a }), new S.ImportNamespaceSpecifier({ local: b })] }),
  "import type a, * as b from 'm';"],
  [new S.ExportAllDeclaration({ exportKind: "type", source: lit("'m'"), attributes: [
    new S.ImportAttribute({ key: n("type"), value: lit("'json'") })] }), "export type * from 'm' with { type: 'json' };"],
  [new S.TSImportEqualsDeclaration({ importKind: "type", id: a, moduleReference: n("b") }), "import type a = b;"],
  [new S.ExportNamedDeclaration({ exportKind: "value", declaration: new S.ClassDeclaration({
    decorators: [new S.Decorator({ expression: a })], id: n("C"), body: new S.ClassBody() }) }), "@a\nexport class C {}"],
  [new S.ReturnStatement({ argument: new S.SequenceExpression({ expressions: [a, b] }) }), "return a, b;"],
  [new S.ThrowStatement({ argument: a }), "throw a;"], [new S.ContinueStatement(), "continue;"],
  [new S.BreakStatement({ label: a }), "break a;"],
  [new S.LabeledStatement({ label: a, body: new S.LabeledStatement({ label: b, body: empty() }) }), "a: b: ;"],
  [new S.WithStatement({ object: a, body: statement(b) }), "with (a)\n    b;"],
];
for (const [node, text] of texts) {
  const standard = node.kind().KIND.startsWith("JSX") ? TSX : TS;
  equal(standard.print(unshare(node)), text);
}
assert(new Printer().print(annotation(T)) === "T");
console.log(texts.length, "kinds");

## TSPRT-05 · Layout: blocks, `else if`, members, comments, and functions within expressions

In [ ]:
const block = (...body: F.SyntaxNode[]) => new S.BlockStatement({ body });
const say = (text: string) => statement(n(text));
const show = (node: F.SyntaxNode) => TS.print(unshare(node));

const nested = new S.IfStatement({ test: a, consequent: block(say("b")), alternate: new S.IfStatement({ test: b,
  consequent: say("c"), alternate: block(new S.IfStatement({ test: c, consequent: block(), alternate: say("d") })) }) });
equal(show(new S.Program({ body: [nested] })), `if (a) {
    b;
} else if (b)
    c;
else {
    if (c) {} else
        d;
}
`);
equal(show(new S.IfStatement({ test: a, consequent: say("b"), alternate: block() })), "if (a)\n    b;\nelse {}");
equal(show(new S.IfStatement({ test: a, consequent: block() })), "if (a) {}");
equal(show(new S.TryStatement({ block: block(), finalizer: block(say("b")) })), "try {} finally {\n    b;\n}");
equal(show(new S.ClassDeclaration({ id: n("C"), superClass: b, body: new S.ClassBody() })), "class C extends b {}");
equal(show(new S.DoWhileStatement({ body: say("a"), test: b })), "do\n    a;\nwhile (b);");
const inner = new S.VariableDeclaration({ declarationKind: "const", declarations: [new S.VariableDeclarator({ id: a,
  init: new S.ArrowFunctionExpression({ body: block(new S.ReturnStatement({ argument: new S.ClassExpression({
    body: new S.ClassBody({ body: [new S.MethodDefinition({ methodKind: "method", key: b,
      value: new S.FunctionExpression({ body: block(say("c")) }) })] }) }) })) }) })] });
const fnDeclaration = new S.FunctionDeclaration({ id: n("f"), body: block(inner,
  new S.Comment({ text: " trailing", trailing: true })) });
equal(show(new S.Program({ body: [fnDeclaration] })), `function f() {
    const a = () => {
        return class {
            b() {
                c;
            }
        };
    }; // trailing
}
`);
equal(show(block(new S.Comment({ text: " only", trailing: true }))), "{ // only\n}");
equal(show(block(new S.Comment({ block: true, text: " only ", trailing: true }))), "{ /* only */\n}");
equal(show(new S.TSEnumDeclaration({ id: n("E"), body: new S.TSEnumBody({ members: [
  new S.Comment({ text: " a", trailing: true }), new S.TSEnumMember({ id: a }), new S.Comment({ text: " b" }),
  new S.Comment({ text: " c", trailing: true })] }) })), "enum E { // a\n    a,\n    // b // c\n}");
const literal = new S.TSTypeLiteral({ members: [new S.Comment({ text: " a" }), new S.TSPropertySignature({ key: a })] });
const alias = new S.TSTypeAliasDeclaration({ id: n("X"), typeAnnotation: literal });
equal(show(new S.TSModuleDeclaration({ moduleKind: "namespace", id: n("N"), body: new S.TSModuleBlock({ body: [alias] }) })),
  "namespace N {\n    type X = {\n        // a\n        a;\n    };\n}");
equal(show(new S.Program({ hashbang: "/usr/bin/env node", body: [new S.Comment({ text: " a" }), say("b")] })),
  "#!/usr/bin/env node\n// a\nb;\n");
equal(show(new S.Program({ body: [new S.Comment({ text: " first", trailing: true })] })), "// first\n");
console.log("ok");

## TSPRT-06 · Printing validates the tree and checks it against the standard; JSX's arrow functions

In [ ]:
raises(PrintError, () => show(new S.SequenceExpression({ expressions: [a] })), "a SequenceExpression needs two expressions");
raises(PrintError, () => ECMAScript2025.print(new S.TSAnyKeyword()), "TSAnyKeyword is not ECMAScript, but this is ES2025");
raises(PrintError, () => show(new S.JSXText({ raw: "a" })), "JSXText needs JSX, but this is TypeScript 5.9");
raises(PrintError, () => TSX.print(unshare(new S.TSTypeAssertion({ typeAnnotation: T, expression: a }))),
  "TSTypeAssertion is not allowed with JSX, but this is TypeScript 5.9 with JSX");
const generic = new S.ArrowFunctionExpression({ typeParameters: new S.TSTypeParameterDeclaration({ params: [
  new S.TSTypeParameter({ name: n("X"), default: new S.TSTypeReference({ typeName: n("A"),
    typeArguments: parameters(T) }) })] }), params: [a], body: a });
equal(TSX.print(unshare(generic)), "<X = A<T>,>(a) => a");
equal(show(generic), "<X = A<T>>(a) => a");
const constrained = new S.ArrowFunctionExpression({ typeParameters: new S.TSTypeParameterDeclaration({ params: [
  new S.TSTypeParameter({ name: n("X"), constraint: T })] }), body: a });
equal(TSX.print(unshare(constrained)), "<X extends T>() => a");
const reparsed = TSX.parse(TSX.print(unshare(new S.Program({ body: [statement(generic)] }))));
assert((reparsed.body[0] as any).expression.params.length === 1);
console.log("ok");